# Algoritmo KNN - Ejemplo completo

Este ejemplo utiliza un dataset con características (Xi) como Edad, Ingresos, Historial crediticio y Monto solicitado para predecir la variable objetivo (Y), que indica si un crédito fue aprobado o no aprobado.

**1. Cargar el conjunto de datos**

In [ ]:
import pandas as pd
df = pd.read_csv('Creditos.csv', sep=';') # Leer dataset cargados desde el pc
df.head(10)

**2. Preprocesamiento**

*Codificación:* Se convierte dos características categóricas (`Historial crediticio` y `Aprobado`) a numericas.

In [ ]:
df['Historial crediticio'] = df['Historial crediticio'].map({'malo': 0, 'regular': 1, 'bueno': 2})
df['Aprobado'] = df['Aprobado'].map({'no': 0, 'si': 1})

**3. Separar variables**

In [ ]:
X = df[['Edad', 'Ingresos', 'Historial crediticio', 'Monto solicitado']]
y = df['Aprobado']


**4. Dividir en entrenamiento y prueba**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state= 50 )

**5. Normalizar o escalar características**

¿Por qué es necesario escalar en KNN?
KNN clasifica los puntos de datos basándose en distancias (por ejemplo, Euclidiana o Manhattan). Si las características están en escalas diferentes, las variables con mayor rango tendrán más influencia en el cálculo de la distancia, lo que puede llevar a resultados sesgados.

Para esto usaremos la clase `StandardScaler`.Esta transforma los datos para que tengan:

*   Media 0
*   Desviación estándar 1

Es decir, realiza la transformación:

𝑋𝑒𝑠𝑐𝑎𝑙𝑎𝑑𝑜= (𝑋−𝜇)/𝜎

Donde:

*   μ es la media de cada característica.
*   σ es la desviación estándar.

Esto garantiza que todas las características contribuyan equitativamente al cálculo de distancia en KNN.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**6. Entrenar el modelo KNN**

Se utiliza la clase `KNeighborsClassifier` para definir el modelo KNN y se toma K = 3.

Se toma la métrica por defecto: `minkowski con p=2` que equivale a la **Euclidiana**.

Se usa de la siguiente manera si se desea una metrica diferente como por ejemplo `Manhattan`:

* model = KNeighborsClassifier(n_neighbors=3, metric='manhattan')


Cuando se quiere usar la distancia de Minkowski con p=3 (ni Euclidiana ni Manhattan)
* model = KNeighborsClassifier(n_neighbors=3, metric='minkowski', p=3)
* Define una distancia intermedia entre Manhattan (lineal) y Euclidiana (cuadrática).
* Penaliza más las grandes diferencias en las coordenadas que la Euclidiana.



In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# Inicializar el modelo
model = KNeighborsClassifier(n_neighbors = 3) # métrica Euclidiana
# Entrenar el modelo
#model.fit(X_train, y_train)
model.fit(X_train_scaled, y_train)

Votación y manejo de empates:
* En KNN, la clase se asigna según la mayoría de votos de los vecinos más cercanos.Si ocurre un empate, se pueden usar tres enfoques: Selección aleatoria, Votación ponderada por distancia ó Reconsiderar los vecinos más próximos.

Ejemplo de votación ponderada
* knn_weighted = KNeighborsClassifier(n_neighbors=3, weights='distance')

**7. Predecir sobre el dataset `test` y evaluar modelo**

In [ ]:
from sklearn.metrics import accuracy_score

#y_pred = model.predict(X_test) # Para cada punto en X_test, calcula la distancia (por defecto, euclidiana) a todos los puntos en X_train.
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitud del modelo: {accuracy:.2f}")

**8. Visualización**

In [ ]:
import matplotlib.pyplot as plt

# Colores para aprobado y no aprobado
colors = ['red' if aprobado == 0 else 'green' for aprobado in y_test]

# Gráfico de dispersión de los datos de prueba
plt.figure(figsize=(10, 6))
plt.scatter(X_test['Edad'], X_test['Monto solicitado'], c=colors, label='Muestra', s=80, edgecolors='k')

plt.xlabel('Edad')
plt.ylabel('Monto solicitado')
plt.title('Clasificación de Aprobación de Créditos (KNN)')
plt.grid(True)
plt.legend()
plt.show()


**9. Demostrar predicción sobre nuevos datos**

In [ ]:
nuevos_datos = pd.DataFrame({
    'Edad': [30, 50, 28],
    'Ingresos': [4000, 2000, 8000],
    'Historial crediticio': ['bueno', 'malo', 'regular'],
    'Monto solicitado': [5000, 12000, 7000]
})

In [ ]:
# Preprocesamiento
nuevos_datos['Historial crediticio'] = nuevos_datos['Historial crediticio'].map({'malo': 0, 'regular': 1, 'bueno': 2})

In [ ]:
# Utilizar el mismo escalador ajustado con X_train
nuevos_datos_scaled = scaler.transform(nuevos_datos)
print(nuevos_datos_scaled)

In [ ]:
# Realizar predicción
nuevas_predicciones = model.predict(nuevos_datos_scaled)
print(nuevas_predicciones)

In [ ]:
# Mostrar resultados
nuevos_datos['Aprobado'] = ['si' if p == 1 else 'no' for p in nuevas_predicciones]
nuevos_datos['Historial crediticio'] = [
    'malo' if p == 0 else 'regular' if p == 1 else 'bueno'
    for p in nuevos_datos['Historial crediticio']
]
nuevos_datos